In [1]:
%pip install opencv-python


   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.0 MB ? eta -:--:--
   ---- ----------------------------------- 3.9/39.0 MB 12.4 MB/s eta 0:00:03
   -------- ------------------------------- 7.9/39.0 MB 15.2 MB/s eta 0:00:03
   ------------ --------------------------- 12.1/39.0 MB 17.2 MB/s eta 0:00:02
   ---------------- ----------------------- 16.3/39.0 MB 17.4 MB/s eta 0:00:02
   -------------------- ------------------- 20.4/39.0 MB 17.7 MB/s eta 0:00:02
   -------------------------- ------------- 25.4/39.0 MB 18.3 MB/s eta 0:00:01
   ------------------------------ --------- 30.1/39.0 MB 18.9 MB/s eta 0:00:01
   ------------------------------------ --- 35.4/39.0 MB 19.6 MB/s eta 0:00:01
   ---------------------------------------  38.8/39.0 MB 19.9 MB/s eta 0:00:01
   ---------------------------------------- 39.0/39.0 MB 19.1 MB/s  0:00:02
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   -

In [2]:
import cv2
import os


def extract_frames(video_path, output_dir="frames", fps_interval=0.5):
    """
    Extract frames from video every `fps_interval` seconds.

    Args:
        video_path (str): Path to the video file.
        output_dir (str): Directory to save extracted frames.
        fps_interval (int): Interval in seconds to extract frames.

    Returns:
        List of (frame_path, timestamp) tuples.
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * fps_interval)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    extracted = []
    frame_idx = 0
    frame_number = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            timestamp = frame_idx / fps
            filename = f"frame_{frame_number:03d}_{int(timestamp)}s.jpg"
            path = os.path.join(output_dir, filename)
            cv2.imwrite(path, frame)
            extracted.append((path, timestamp))
            frame_number += 1

        frame_idx += 1

    cap.release()
    print(f"[INFO] Extracted {len(extracted)} frames from video.")
    return extracted

In [3]:
if __name__ == "__main__":
    video_file = "road.mp4"
    frames = extract_frames(
        video_file, output_dir="video_frames", fps_interval=0.5)

    for fpath, time in frames:
        print(f"Frame at {time:.2f}s saved to: {fpath}")

[INFO] Extracted 28 frames from video.
Frame at 0.00s saved to: video_frames\frame_000_0s.jpg
Frame at 0.50s saved to: video_frames\frame_001_0s.jpg
Frame at 1.00s saved to: video_frames\frame_002_1s.jpg
Frame at 1.50s saved to: video_frames\frame_003_1s.jpg
Frame at 2.00s saved to: video_frames\frame_004_2s.jpg
Frame at 2.50s saved to: video_frames\frame_005_2s.jpg
Frame at 3.00s saved to: video_frames\frame_006_3s.jpg
Frame at 3.50s saved to: video_frames\frame_007_3s.jpg
Frame at 4.00s saved to: video_frames\frame_008_4s.jpg
Frame at 4.50s saved to: video_frames\frame_009_4s.jpg
Frame at 5.00s saved to: video_frames\frame_010_5s.jpg
Frame at 5.50s saved to: video_frames\frame_011_5s.jpg
Frame at 6.00s saved to: video_frames\frame_012_6s.jpg
Frame at 6.50s saved to: video_frames\frame_013_6s.jpg
Frame at 7.00s saved to: video_frames\frame_014_7s.jpg
Frame at 7.50s saved to: video_frames\frame_015_7s.jpg
Frame at 8.00s saved to: video_frames\frame_016_8s.jpg
Frame at 8.50s saved to: v

In [7]:
# ollama model run
%pip install ollama

  Using cached ollama-0.5.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached ollama-0.5.1-py3-none-any.whl (13 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 10.9 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.1-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   -------- -------------------------------  2/10 [pydantic-core]
   ------------ ---------------------------  3/10 [h11]
   ---------------- -----------------------  4/10 [annotated-types]
   -------------------- -------------------  5/10 [pyda

In [8]:
import ollama
ollama_model = "llava:latest"

In [14]:
# image to base64 conversion
import base64
def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


In [1]:
import ollama

system_prompt = """
You are an expert Traffic Police Officer assigned to review traffic surveillance footage for rule violations. 
Your job is to carefully analyze the given image and report **all visible traffic violations**.

For each violation, include:
- The type of violation
- A short explanation
- The object(s) involved (e.g., vehicle type, pedestrian, signal)
- If possible, approximate position (e.g., left/right/center)

You must act as an observant, unbiased traffic inspector. Only report violations you can visually confirm. If no violation is clearly visible, say so.

Violation examples include (but are not limited to):
- Running a red light
- Pedestrian jaywalking
- Helmet not worn on bike
- Triple riding on two-wheelers
- Vehicles over zebra crossing during red
- Driving in wrong lane
- Using a mobile phone while driving

Use clear, concise language. Mention all violations present in the image. Be truthful and avoid guessing.
"""


response = ollama.chat(
    model='llava',
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": "Please review this image and report all traffic rule violations.",
            "images": ["video_frames/frame_008_4s.jpg"]
        }
    ]
)

print(response['message']['content'])

 The image provided shows a street scene with several potential traffic violations:

1. The vehicle appears to be crossing a red signal, which is an illegal maneuver as it goes against the flow of traffic and could cause collisions.
2. The car's license plate is obscured, but if it were visible and not legally covered, this would constitute a violation of not displaying required registration information.
3. There are multiple persons in the image who do not appear to be following pedestrian safety rules by crossing at a zebra crossing during what seems to be a red light for vehicles. This is considered jaywalking.
4. The presence of a racing flag or similar item on the vehicle might indicate speeding, which can be dangerous and illegal in most places.
5. If the car's taillights are not functioning properly, this could constitute a violation of having non-functional rear lights.
6. There is no clear indication of the rider wearing proper protective gear such as a helmet while riding on 